![image_1780908244748.png](./image_1780908244748.png "image_1780908244748.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import Window
# Initialize Spark session
spark = SparkSession.builder.appName("MonthlyCardsIssued").getOrCreate()

# Define the dataset as a list of tuples
data = [
    ("Chase Sapphire Reserve", 170000, 1, 2021),
    ("Chase Sapphire Reserve", 175000, 2, 2021),
    ("Chase Sapphire Reserve", 180000, 3, 2021),
    ("Amex Gold", 250000, 7, 2022),
    ("Amex Gold", 230000, 8, 2022),
    ("Capital One Venture", 100000, 3, 2021),
    ("Capital One Venture", 120000, 4, 2021),
]

# Define the schema (column names)
columns = ["card_name", "issued_amount", "issue_month", "issue_year"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

# Show the DataFrame
df.show(truncate=False)


In [0]:
result_df = (
    df.withColumn(
        "rn",
        f.dense_rank().over(
            Window.partitionBy("card_name").orderBy(
                f.col("issue_year").asc(), f.col("issue_month").asc()
            )
        ),
    )
    .filter(f.col("rn") == 1)
    .select("card_name", "issued_amount")
    .orderBy(f.col("issued_amount").desc())
)
display(result_df)